# 🚀 Mermaid Diagram Generator – Gemma-3-1B-IT + **Unsloth** (2× faster, 60% less VRAM)

**Optimized for Colab T4 (16 GB)**
- 4-bit NF4 quantization
- LoRA rank 16 targeting **all linear modules** (perfect for rigid Mermaid syntax)
- Unsloth 2× faster training + lower memory
- Full train → merge → INT8 .tflite for LiteRT edge deployment
- Custom Mermaid validation via `mmdc` (with sandbox fix)

Run cells sequentially. Training ≈ **45 min–1 hour** on T4.

In [ ]:
%%capture
!pip install -q --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-deps
!pip install -q unsloth_zoo
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q ai-edge-torch-nightly

# Node LTS and Mermaid CLI (Skip when need to only train)
!curl -fsSL https://deb.nodesource.com/setup_24.x | sudo -E bash -
!apt-get install -y nodejs -qq
!apt-get update -qq && apt-get install -y chromium-browser libnss3 libgconf-2-4 libfontconfig1 -qq
!npm install -g @mermaid-js/mermaid-cli --silent

import torch
import ast
import subprocess
import os
import json
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

In [ ]:
# ===========================
# 2. Data Loading & Formatting
# ===========================
import re
import ast

# model_id = "google/gemma-3-1b-it"
model_id = "unsloth/gemma-3-1b-it-bnb-4bit" # 4-bit NF4 quantization

dataset = load_dataset("colinfrisch/diagrams-mermaid-filtered", split="train")
split_dataset = dataset.train_test_split(test_size=1000, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

def format_row(example):
    try:
        # The dataset already provides 'messages' as a list of dicts
        messages = example["messages"]

        # Ensure it is a list (handles cases where it might be a string in some versions)
        if isinstance(messages, str):
            messages = ast.literal_eval(messages)

        for msg in messages:
            if msg.get("role") == "assistant":
                msg["role"] = "model"

        formatted_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": formatted_text}
    except Exception as e:
        return {"text": None}

# Initialize model and tokenizer (temporary for formatting only)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

train_dataset = train_dataset.map(format_row, remove_columns=train_dataset.column_names)
train_dataset = train_dataset.filter(lambda x: x["text"] is not None)

print(f"Training dataset size after filtering: {len(train_dataset)}")
if len(train_dataset) > 0:
    print("Example formatted text:")
    print(train_dataset[0]["text"][:500])
else:
    print("❌ Still empty. Check if tokenizer.apply_chat_template is failing.")

In [ ]:
# ===========================
# 3. Unsloth + QLoRA Setup (4-bit NF4 + all-linear)
# ===========================
max_seq_length = 2048
dtype = None  # Auto-detect (bfloat16 on T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

In [ ]:
# ===========================
# 4. Training with SFTTrainer (GPU Optimized)
# ===========================
training_args = SFTConfig(
    output_dir="./gemma-mermaid-unsloth-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,                             # Enable fp16 for T4 GPU
    bf16=False,                            # T4 does not support bf16
    max_seq_length=max_seq_length,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()
print("✅ Unsloth training completed on GPU")

In [ ]:
# ===========================
# 5. Custom Mermaid Validation
# ===========================
def validate_mermaid(mmd_code, filename="test.mmd"):
    with open(filename, "w") as f:
        f.write(mmd_code)
    try:
        config_path = "puppeteer-config.json"
        with open(config_path, "w") as f:
            json.dump({"args": ["--no-sandbox", "--disable-setuid-sandbox", "--disable-dev-shm-usage"]}, f)
        result = subprocess.run(
            ["mmdc", "-i", filename, "-o", "test.svg", "-p", config_path],
            capture_output=True,
            text=True,
            timeout=30
        )
        return result.returncode == 0, result.stdout + result.stderr
    except Exception as e:
        return False, str(e)

# Best fix for Gemma-3: Patch the attention mask logic via Unsloth helper
# (Required for Gemma-3 to avoid "attention_mask is None" / NoneType errors in Gemma3Attention_forward)
FastLanguageModel.for_inference(model)

success_count = 0
test_samples = eval_dataset.select(range(min(20, len(eval_dataset))))

for idx, example in enumerate(test_samples):
    try:
        messages = example["messages"]
        user_content = messages[0]["content"]

        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": user_content}],
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            # Setting use_cache=False is a fallback if broadcasting errors persist
            # but we try True first with patched model
            outputs = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=512,
                use_cache=True,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        input_length = inputs.input_ids.shape[1]
        generated = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        mmd = None
        if "```mermaid" in generated:
            start = generated.find("```mermaid") + 11
            end = generated.find("```", start)
            mmd = generated[start:end].strip()

        if mmd:
            valid, log = validate_mermaid(mmd)
            success_count += int(valid)
            print(f"Sample {idx+1}/20 | Valid: {valid}")
            if not valid:
                 print(f"   Error log excerpt: {log[:100]}...")
        else:
            print(f"Sample {idx+1}/20 | No Mermaid block found")
    except Exception as e:
        print(f"Sample {idx+1}/20 | Error: {e}")

print(f"\n✅ Mermaid compilation success rate: {success_count}/20 = {success_count/20*100:.1f}%")

In [ ]:
# ===========================
# 6. Merge LoRA → Standalone PyTorch (Unsloth fast merge)
# ===========================
model.save_pretrained_merged(
    "gemma-mermaid-merged",
    tokenizer,
    save_method="merged_16bit"   # full 16-bit weights, ready for edge conversion
)
print("✅ LoRA merged into standalone PyTorch model at ./gemma-mermaid-merged")

In [ ]:
# ===========================
# 6.5. MacOS Version of Step 5
# ===========================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./gemma-mermaid-merged"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,   # or bfloat16 if your Mac supports it
    device_map="mps",            # ← this is the key change
    trust_remote_code=True
)
model.eval()

# ... rest of your validation loop stays almost the same, just change device
inputs = tokenizer(prompt, return_tensors="pt").to("mps")
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

In [ ]:
# ===========================
# 7. Conversion to .tflite (INT8 dynamic-range via ai-edge-torch GenAI)
# ===========================
import ai_edge_torch.genai.torch_models as edge_models
import ai_edge_torch.genai as genai

model_path = "./gemma-mermaid-merged"

try:
    # NOTE: Gemma-3 support in ai-edge-torch is limited/newer; may require update or fallback
    # For Gemma-3-1B you may need a different class or custom conversion (check ai-edge-torch docs)
    # Keeping Gemma2 class as placeholder – replace with Gemma3 equivalent if available
    edge_gemma = edge_models.gemma2.Gemma2(model_path)

    quant_config = genai.quantize.QuantConfig(
        weight_quant=genai.quantize.DynamicInt8WeightQuant()
    )

    print("🚀 Starting GenAI conversion (may take significant RAM/time)...")
    converter = genai.Converter(edge_gemma, quant_config)
    tflite_model = converter.convert()

    with open("gemma_mermaid_diagram_generator.tflite", "wb") as f:
        f.write(tflite_model)
    print("✅ Successfully converted to gemma_mermaid_diagram_generator.tflite (INT8)")
except Exception as e:
    print(f"❌ Conversion failed (common on T4 due to RAM or Gemma-3 compatibility). Error: {e}")
    print("Note: Gemma-3-1B GenAI conversion often requires updated ai-edge-torch or 50GB+ RAM → use Colab Pro or run locally.")